In [ ]:
import torch
import json
from pathlib import Path
from typing import List, Dict, Iterator, Any
from datetime import datetime
from transformers import RobertaTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


67-68 for fine tuning

## Batch Cache

In [ ]:
class SimpleBatchCache:
    """
    Simple batch cache with master.json metadata tracking.
    Stores batches as batch001.pt, batch002.pt, etc.
    Tracks metadata in master.json for easy monitoring.
    """

    def __init__(self, cache_dir: str = "./batch_cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)

        # Master metadata file
        self.master_file = self.cache_dir / "master.json"
        self.metadata = self._load_metadata()

        # Find the next available batch number
        self.next_batch_num = self._get_next_batch_number()

    def _load_metadata(self) -> Dict[str, Any]:
        """Load or initialize master metadata."""
        if self.master_file.exists():
            with open(self.master_file, 'r') as f:
                return json.load(f)
        else:
            # Initialize fresh metadata
            return {
                "created_at": datetime.now().isoformat(),
                "last_updated": datetime.now().isoformat(),
                "total_files": 0,
                "total_batches": 0,
                "files": {},
                "stats": {
                    "total_tokens": 0,
                    "total_sequences": 0,
                    "avg_sequence_length": 0
                }
            }

    def _save_metadata(self):
        """Save metadata to master.json."""
        self.metadata["last_updated"] = datetime.now().isoformat()
        with open(self.master_file, 'w') as f:
            json.dump(self.metadata, f, indent=2)

    def _get_next_batch_number(self) -> int:
        """Find the highest batch number from metadata or files."""
        if self.metadata["files"]:
            return max(int(k) for k in self.metadata["files"].keys()) + 1
        else:
            # Fallback to file scanning if metadata is empty
            existing_files = list(self.cache_dir.glob("batch*.pt"))
            if not existing_files:
                return 1
            numbers = []
            for file in existing_files:
                try:
                    num_str = file.stem.replace("batch", "")
                    numbers.append(int(num_str))
                except ValueError:
                    continue
            return max(numbers) + 1 if numbers else 1

    def _calculate_batch_stats(self, batches: List[Dict[str, torch.Tensor]]) -> Dict[str, Any]:
        """Calculate statistics for a set of batches."""
        total_tokens = 0
        total_sequences = 0

        for batch in batches:
            if 'input_ids' in batch:
                batch_size, seq_len = batch['input_ids'].shape
                total_sequences += batch_size
                total_tokens += batch_size * seq_len

        return {
            "num_batches": len(batches),
            "total_sequences": total_sequences,
            "total_tokens": total_tokens,
            "avg_sequence_length": total_tokens / total_sequences if total_sequences > 0 else 0
        }

    def save_batches(self, batches: List[Dict[str, torch.Tensor]],
                    metadata: Dict[str, Any] = None) -> str:
        """
        Save batches with metadata tracking.

        Args:
            batches: List of batch dictionaries
            metadata: Additional metadata about these batches

        Returns:
            Path to saved file
        """
        filename = f"batch{self.next_batch_num:03d}.pt"
        filepath = self.cache_dir / filename

        # Save batches to file
        torch.save(batches, filepath)

        # Calculate batch statistics
        batch_stats = self._calculate_batch_stats(batches)

        # Update master metadata
        file_metadata = {
            "filename": filename,
            "created_at": datetime.now().isoformat(),
            "num_batches": len(batches),
            **batch_stats
        }

        # Add custom metadata if provided
        if metadata:
            file_metadata["custom"] = metadata

        # Update master metadata
        self.metadata["files"][str(self.next_batch_num)] = file_metadata
        self.metadata["total_files"] = len(self.metadata["files"])
        self.metadata["total_batches"] += len(batches)

        # Update global stats
        self.metadata["stats"]["total_tokens"] += batch_stats["total_tokens"]
        self.metadata["stats"]["total_sequences"] += batch_stats["total_sequences"]
        if self.metadata["stats"]["total_sequences"] > 0:
            self.metadata["stats"]["avg_sequence_length"] = (
                self.metadata["stats"]["total_tokens"] /
                self.metadata["stats"]["total_sequences"]
            )

        # Save metadata
        self._save_metadata()

        print(f"💾 Saved {len(batches)} batches to {filename}")
        print(f"📊 Stats: {batch_stats['total_sequences']} sequences, {batch_stats['total_tokens']} tokens")

        # Increment for next file
        self.next_batch_num += 1

        return str(filepath)

    def save_single_batch(self, batch: Dict[str, torch.Tensor],
                         metadata: Dict[str, Any] = None) -> str:
        """Save a single batch with metadata."""
        return self.save_batches([batch], metadata)

    def load_batches(self, filename: str) -> List[Dict[str, torch.Tensor]]:
        """Load batches from a file."""
        return torch.load(self.cache_dir / filename)

    def get_file_metadata(self, batch_num: int) -> Dict[str, Any]:
        """Get metadata for a specific batch file."""
        return self.metadata["files"].get(str(batch_num), {})

    def get_all_files(self) -> List[str]:
        """Get all batch files in sequential order with metadata."""
        files = []
        for batch_num in sorted(self.metadata["files"].keys(), key=int):
            file_info = self.metadata["files"][batch_num]
            files.append(file_info["filename"])
        return files

    def get_all_files_with_metadata(self) -> List[Dict[str, Any]]:
        """Get all files with their metadata."""
        return [
            {**self.metadata["files"][batch_num], "batch_number": int(batch_num)}
            for batch_num in sorted(self.metadata["files"].keys(), key=int)
        ]

    def iter_all_batches(self) -> Iterator[Dict[str, torch.Tensor]]:
        """Iterate through all batches in all files."""
        for file_path in self.get_all_files():
            batches = self.load_batches(file_path)
            for batch in batches:
                yield batch

    def iter_files_with_metadata(self) -> Iterator[tuple[List[Dict[str, torch.Tensor]], Dict[str, Any]]]:
        """Iterate through files with their metadata."""
        for file_info in self.get_all_files_with_metadata():
            batches = self.load_batches(file_info["filename"])
            yield batches, file_info

    def clear_cache(self):
        """Clear all cached files and reset metadata."""
        for file in self.cache_dir.glob("batch*.pt"):
            file.unlink()

        if self.master_file.exists():
            self.master_file.unlink()

        # Reset
        self.metadata = self._load_metadata()  # This will create fresh metadata
        self.next_batch_num = 1

        print("🧹 Batch cache cleared")

    def get_stats(self) -> Dict[str, Any]:
        """Get comprehensive cache statistics."""
        files = self.get_all_files()
        total_batches = self.metadata["total_batches"]

        return {
            **self.metadata["stats"],
            "total_files": len(files),
            "total_batches": total_batches,
            "cache_dir": str(self.cache_dir),
            "created_at": self.metadata["created_at"],
            "last_updated": self.metadata["last_updated"]
        }

    def print_detailed_stats(self):
        """Print detailed statistics about the cache."""
        stats = self.get_stats()
        files_info = self.get_all_files_with_metadata()

        print("=" * 50)
        print("BATCH CACHE STATISTICS")
        print("=" * 50)
        print(f"Total files: {stats['total_files']}")
        print(f"Total batches: {stats['total_batches']}")
        print(f"Total sequences: {stats['total_sequences']:,}")
        print(f"Total tokens: {stats['total_tokens']:,}")
        print(f"Avg sequence length: {stats['avg_sequence_length']:.1f}")
        print(f"Cache directory: {stats['cache_dir']}")
        print(f"Created: {stats['created_at']}")
        print(f"Last updated: {stats['last_updated']}")
        print("\nFile Details:")
        for file_info in files_info:
            print(f"  {file_info['filename']}: {file_info['num_batches']} batches, "
                  f"{file_info['total_sequences']} sequences, {file_info['total_tokens']} tokens")
        print("=" * 50)

    def update_file_metadata(self, batch_num: int, new_metadata: Dict[str, Any]):
        """Update metadata for a specific batch file."""
        if str(batch_num) in self.metadata["files"]:
            self.metadata["files"][str(batch_num)].update(new_metadata)
            self._save_metadata()

    def delete_batch_file(self, batch_num: int):
        """
        Deletes a specific batch file and its metadata.

        Args:
            batch_num (int): The batch number of the file to delete (e.g., 5 for batch005.pt).
        """
        batch_num_str = str(batch_num)
        if batch_num_str not in self.metadata["files"]:
            print(f"❌ Batch {batch_num_str} not found in metadata. Nothing to delete.")
            return

        file_metadata_to_delete = self.metadata["files"][batch_num_str]
        filename_to_delete = file_metadata_to_delete["filename"]
        filepath_to_delete = self.cache_dir / filename_to_delete

        # 1. Update global statistics in metadata
        self.metadata["total_files"] -= 1
        self.metadata["total_batches"] -= file_metadata_to_delete["num_batches"]
        self.metadata["stats"]["total_sequences"] -= file_metadata_to_delete["total_sequences"]
        self.metadata["stats"]["total_tokens"] -= file_metadata_to_delete["total_tokens"]

        if self.metadata["stats"]["total_sequences"] > 0:
            self.metadata["stats"]["avg_sequence_length"] = (
                self.metadata["stats"]["total_tokens"] /
                self.metadata["stats"]["total_sequences"]
            )
        else:
            self.metadata["stats"]["avg_sequence_length"] = 0

        # 2. Remove the entry from files metadata
        del self.metadata["files"][batch_num_str]

        # 3. Save updated metadata
        self._save_metadata()

        # 4. Delete the physical file
        if filepath_to_delete.exists():
            filepath_to_delete.unlink()
            print(f"🗑️ Successfully deleted file '{filename_to_delete}' and its metadata.")
        else:
            print(f"⚠️ Metadata for '{filename_to_delete}' deleted, but file was not found on disk.")

        # Recalculate next_batch_num in case the deleted batch was the highest one
        self.next_batch_num = self._get_next_batch_number()

## Layer and clinet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import RobertaTokenizer

In [ ]:
from typing import List, Dict
from transformers import RobertaTokenizer
import torch

class QADataProcessor:
    """
    Processes Q&A data from text files using RobertaTokenizer.
    Handles the specific format: "Number.Client: ... Lawyer: ..."
    """

    def __init__(self, tokenizer_name: str = "roberta-base", max_length: int = 512):
        self.tokenizer = RobertaTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length

        # Add padding token if it doesn't exist
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def parse_qa_text(self, text: str) -> List[Dict[str, str]]:
        """
        Parse Q&A text in the format:
        "99.Client: Do these constitutional instruments promote stability?
        Lawyer: Absolutely—they resolve jurisdictional conflicts..."

        Returns:q
            List of dictionaries with 'client' and 'lawyer' fields
        """
        qa_pairs = []
        lines = text.strip().split('\n')

        current_client = None
        current_lawyer = None

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Check if line starts with number.Client:
            if '.' in line and 'Client:' in line:
                # Save previous pair if exists
                if current_client is not None and current_lawyer is not None:
                    qa_pairs.append({
                        'client': current_client,
                        'lawyer': current_lawyer
                    })

                # Start new pair
                parts = line.split('Client:', 1)
                current_client = parts[1].strip() if len(parts) > 1 else ""
                current_lawyer = None

            # Check if line starts with Lawyer:
            elif line.startswith('Lawyer:'):
                if current_client is not None:
                    current_lawyer = line.replace('Lawyer:', '').strip()

            # Continuation of lawyer's answer
            elif current_lawyer is not None and line:
                current_lawyer += " " + line

        # Don't forget the last pair
        if current_client is not None and current_lawyer is not None:
            qa_pairs.append({
                'client': current_client,
                'lawyer': current_lawyer
            })

        return qa_pairs

    def format_for_training(self, qa_pair: Dict[str, str]) -> str:
        """
        Format Q&A pair for causal language model training.
        Uses special tokens to separate client and lawyer parts.
        """
        client_text = qa_pair['client']
        lawyer_text = qa_pair['lawyer']

        # Format: Client: [question] </s> Lawyer: [answer]
        formatted = f"Client: {client_text} </s> Lawyer: {lawyer_text}"
        return formatted

    def tokenize_qa_pairs(self, qa_pairs: List[Dict[str, str]]) -> List[Dict[str, torch.Tensor]]:
        """
        Tokenize Q&A pairs for training.
        Returns list of tokenized examples ready for batch creation.
        Labels are shifted versions of input_ids for standard causal language modeling,
        with padding tokens masked with -100.
        """
        tokenized_examples = []

        for qa_pair in qa_pairs:
            formatted_text = self.format_for_training(qa_pair)

            # Tokenize the full sequence
            tokenized = self.tokenizer(
                formatted_text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            input_ids = tokenized['input_ids'].squeeze(0)
            attention_mask = tokenized['attention_mask'].squeeze(0)

            # Create labels by shifting input_ids to the left
            # labels[i] should be input_ids[i+1]
            labels = input_ids.clone()
            # Shift labels left by one, and set the last token to -100
            labels = torch.cat((labels[1:], torch.tensor([-100], dtype=torch.long)))

            # Ensure labels have the same length as input_ids and attention_mask
            # This handles cases where input_ids might be shorter than max_length due to truncation, or padding
            if len(labels) < self.max_length:
                padding_needed = self.max_length - len(labels)
                labels = torch.cat((labels, torch.full((padding_needed,), -100, dtype=torch.long)))
            elif len(labels) > self.max_length: # Should not happen if truncation is handled by tokenizer
                labels = labels[:self.max_length]

            # Mask out padding tokens in labels with -100 (where attention_mask is 0)
            # This is important because the shifted labels might still contain pad_token_id
            labels = torch.where(attention_mask == 0, torch.tensor(-100, dtype=torch.long), labels)

            example = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            }

            tokenized_examples.append(example)

        return tokenized_examples
    def create_batches(self, examples: List[Dict[str, torch.Tensor]], batch_size: int = 4) -> List[Dict[str, torch.Tensor]]:
        """
        Create batches from individual examples.
        """
        batches = []

        for i in range(0, len(examples), batch_size):
            batch_examples = examples[i:i + batch_size]

            # Stack individual examples into batch tensors
            batch = {
                'input_ids': torch.stack([ex['input_ids'] for ex in batch_examples]),
                'attention_mask': torch.stack([ex['attention_mask'] for ex in batch_examples]),
                'labels': torch.stack([ex['labels'] for ex in batch_examples])
            }

            batches.append(batch)

        return batches

In [ ]:
def process_qa_file_to_cache(input_file: str, cache_dir: str = "./qa_batch_cache",
                           batch_size: int = 16, max_length: int = 512):
    """
    Complete pipeline: Read Q&A file -> Parse -> Tokenize -> Batch -> Cache
    """
    # Initialize processor and cache
    processor = QADataProcessor(max_length=max_length)
    cache = SimpleBatchCache(cache_dir)

    # Read input file
    print(f"📖 Reading Q&A data from: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        text_data = f.read()

    # Parse Q&A pairs
    print("🔍 Parsing Q&A pairs...")
    qa_pairs = processor.parse_qa_text(text_data)
    print(f"✅ Found {len(qa_pairs)} Q&A pairs")

    # Tokenize examples
    print("🔤 Tokenizing examples...")
    tokenized_examples = processor.tokenize_qa_pairs(qa_pairs)
    print(f"✅ Tokenized {len(tokenized_examples)} examples")

    # Create batches
    print("📦 Creating batches...")
    batches = processor.create_batches(tokenized_examples, batch_size=batch_size)
    print(f"✅ Created {len(batches)} batches (batch_size={batch_size})")

    # Save all batches to cache in a single file with metadata
    print("💾 Saving all batches to a single cache file...")
    cache.save_batches(batches, {
        "source_file": input_file,
        "data_type": "legal_qa",
        "total_batches_in_file": len(batches),
        "original_qa_pairs": len(qa_pairs),
        "tokenizer": "roberta-base",
        "max_length": max_length
    })

    # Print final statistics
    print("\n" + "="*60)
    print("PROCESSING COMPLETE")

In [ ]:
process_qa_file_to_cache('/content/client_lawer.txt' , "/content/drive/MyDrive/cache",
                           batch_size= 16, max_length = 250)

📖 Reading Q&A data from: /content/client_lawer.txt
🔍 Parsing Q&A pairs...
✅ Found 1 Q&A pairs
🔤 Tokenizing examples...
✅ Tokenized 1 examples
📦 Creating batches...
✅ Created 1 batches (batch_size=16)
💾 Saving all batches to a single cache file...
💾 Saved 1 batches to batch060.pt
📊 Stats: 1 sequences, 250 tokens

PROCESSING COMPLETE


## Raw Text Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import RobertaTokenizer
from typing import List, Dict
import torch

In [ ]:
from transformers import RobertaTokenizer
from typing import List, Dict
import torch

class SimpleDocumentProcessor:
    """Simple processor for creating overlapping chunks for causal LM training."""

    def __init__(self, tokenizer: RobertaTokenizer, max_length: int =512, overlap: int = 50):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.overlap = overlap

    def create_overlapping_chunks(self, text: str) -> List[str]:
        """Create overlapping chunks of text for causal language modeling."""
        # Tokenize the entire text
        tokens = self.tokenizer.encode(text, add_special_tokens=False)

        chunks = []
        start = 0

        while start < len(tokens):
            # Calculate end position for this chunk
            end = start + self.max_length

            # Extract tokens for this chunk
            chunk_tokens = tokens[start:end]

            # Decode back to text
            chunk_text = self.tokenizer.decode(chunk_tokens, clean_up_tokenization_spaces=True)
            chunks.append(chunk_text)

            # Move start position with overlap
            start += (self.max_length - self.overlap)

            # Break if we're not making progress
            if start >= len(tokens):
                break

        return chunks

    def create_training_examples(self, text: str) -> List[Dict[str, torch.Tensor]]:
        """Create training examples from text for causal language modeling."""
        # Create overlapping chunks
        chunks = self.create_overlapping_chunks(text)

        examples = []

        for chunk in chunks:
            # Tokenize the chunk for causal LM
            encoding = self.tokenizer(
                chunk,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            input_ids = encoding['input_ids'].squeeze(0)
            attention_mask = encoding['attention_mask'].squeeze(0)

            # Create labels by shifting input_ids to the left
            # labels[i] should be input_ids[i+1]
            labels = input_ids.clone()
            # Shift labels left by one, and set the last token to -100
            labels = torch.cat((labels[1:], torch.tensor([-100], dtype=torch.long)))

            # Ensure labels have the same length as input_ids and attention_mask
            if len(labels) < self.max_length:
                padding_needed = self.max_length - len(labels)
                labels = torch.cat((labels, torch.full((padding_needed,), -100, dtype=torch.long)))
            elif len(labels) > self.max_length: # Should not happen if truncation is handled by tokenizer
                labels = labels[:self.max_length]

            # Mask out padding tokens in labels with -100 (where attention_mask is 0)
            labels = torch.where(attention_mask == 0, torch.tensor(-100, dtype=torch.long), labels)

            example = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels
            }

            examples.append(example)

        return examples

    def create_batches(self, examples: List[Dict[str, torch.Tensor]], batch_size: int = 8) -> List[Dict[str, torch.Tensor]]:
        """
        Create batches from individual examples.
        """
        batches = []

        for i in range(0, len(examples), batch_size):
            batch_examples = examples[i:i + batch_size]

            # Stack individual examples into batch tensors
            batch = {
                'input_ids': torch.stack([ex['input_ids'] for ex in batch_examples]),
                'attention_mask': torch.stack([ex['attention_mask'] for ex in batch_examples]),
                'labels': torch.stack([ex['labels'] for ex in batch_examples])
            }

            batches.append(batch)

        return batches

    def process_file(self, file_path: str) -> List[Dict[str, torch.Tensor]]:
        """Process a single file and return training examples."""
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        return self.create_training_examples(text)


In [ ]:
from pathlib import Path
from typing import List

def combine_files(input_files: List[str], output_file: str):
    """
    Combines the content of multiple input text files into a single output file.
    Removes all newline characters from each input file's content, replacing them with a space,
    before combining.

    Args:
        input_files (List[str]): A list of paths to the input text files.
        output_file (str): The path to the output file where combined content will be saved.
    """
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as outfile:
        for i, input_file in enumerate(input_files):
            input_path = Path(input_file)
            if not input_path.exists():
                print(f"Warning: Input file not found: {input_file}. Skipping.")
                continue

            print(f"Reading content from: {input_file}")
            with open(input_path, 'r', encoding='utf-8') as infile:
                # Read content and replace all newline characters with a space
                content = infile.read().replace('\n', ' ')
                outfile.write(content)

    print(f"Successfully combined {len(input_files)} files into {output_file}")

['Dr_Partap_Singh_And_Anr_vs_Director_Of_Enforcement_Foreign_on_26_April_1985_1.PDF', 'Darshan_Singh_Bhasuri_Ors_vs_State_Of_Punjab_on_31_March_1983_1.PDF', 'State_Of_Haryana_And_Ors_vs_Birkha_Ram_And_Ors_on_18_July_1985_1.PDF', 'Khazan_Chand_Etc_vs_State_Of_Jammu_And_Kashmir_And_Others_on_13_February_1984_1.PDF', 'Sital_Prasad_Saxena_Dead_By_Lrs_vs_Union_Of_India_And_Ors_on_28_August_1984_1.PDF', 'Mecleod_Co_Ltd_vs_State_Of_Orissa_Ors_on_23_November_1983_1.PDF', 'State_Of_Haryana_Ors_vs_Lal_Chand_Ors_on_2_May_1984_1.PDF', 'Workmen_Employed_By_Hindustan_Lever_Ltd_vs_Hindustan_Lever_Limited_on_28_August_1984_1.PDF', 'Smt_Dipo_vs_Wassan_Singh_Others_on_5_May_1983_1.PDF', 'Sharvan_Kumar_vs_State_Of_Uttar_Pradesh_on_29_July_1985_1.PDF', 'Keshav_Sitaram_Sali_vs_State_Of_Maharashtra_on_6_January_1983_1.PDF', 'Aher_Pitha_Vajshi_And_Ors_vs_State_Of_Gujarat_on_31_March_1983_1.PDF', 'C_K_Murthy_vs_Government_Of_Andhra_Pradesh_on_28_September_1984_1.PDF', 'Lohia_Machines_Ltd_Anr_vs_Union_Of_India

In [ ]:
input_files_to_combine =  pdf_files
output_combined_file = 'supreme_court_83-85.txt'

In [ ]:
combine_files(input_files_to_combine, output_combined_file)

Reading content from: Dr_Partap_Singh_And_Anr_vs_Director_Of_Enforcement_Foreign_on_26_April_1985_1.PDF
Reading content from: Darshan_Singh_Bhasuri_Ors_vs_State_Of_Punjab_on_31_March_1983_1.PDF
Reading content from: State_Of_Haryana_And_Ors_vs_Birkha_Ram_And_Ors_on_18_July_1985_1.PDF
Reading content from: Khazan_Chand_Etc_vs_State_Of_Jammu_And_Kashmir_And_Others_on_13_February_1984_1.PDF
Reading content from: Sital_Prasad_Saxena_Dead_By_Lrs_vs_Union_Of_India_And_Ors_on_28_August_1984_1.PDF
Reading content from: Mecleod_Co_Ltd_vs_State_Of_Orissa_Ors_on_23_November_1983_1.PDF
Reading content from: State_Of_Haryana_Ors_vs_Lal_Chand_Ors_on_2_May_1984_1.PDF
Reading content from: Workmen_Employed_By_Hindustan_Lever_Ltd_vs_Hindustan_Lever_Limited_on_28_August_1984_1.PDF
Reading content from: Smt_Dipo_vs_Wassan_Singh_Others_on_5_May_1983_1.PDF
Reading content from: Sharvan_Kumar_vs_State_Of_Uttar_Pradesh_on_29_July_1985_1.PDF
Reading content from: Keshav_Sitaram_Sali_vs_State_Of_Maharashtra_on

In [ ]:
def process_legal_documents_simple(file_paths: List[str], cache_dir: str = "./legal_cache", batch_size: int = 8):
    """Process legal documents and store in cache - simple version."""

    # Initialize components
    tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
    processor = SimpleDocumentProcessor(
        tokenizer=tokenizer,
        max_length=2048,
        overlap=50
    )
    cache = SimpleBatchCache(cache_dir)

    all_examples = []

    for file_path in file_paths:
        print(f"📄 Processing {file_path}...")

        # Process file and get training examples
        examples_from_file = processor.process_file(file_path)
        print(f"✅ Created {len(examples_from_file)} training examples")

        # Accumulate all examples
        all_examples.extend(examples_from_file)

    # Create batches from all accumulated examples
    if all_examples:
        print(f"📦 Creating batches from {len(all_examples)} examples...")
        batches_to_save = processor.create_batches(all_examples, batch_size=batch_size)
        print(f"✅ Created {len(batches_to_save)} batches (batch_size={batch_size})")

        # Save all batches to cache in a single file
        print("💾 Saving all batches to cache...")
        cache.save_batches(batches_to_save, metadata={
            "source_files": file_paths,
            "data_type": "legal_document_clm",
            "total_examples_processed": len(all_examples),
            "tokenizer": "roberta-base",
            "max_length": 512,
            "overlap": 50
        })
        print("💾 Saved all processed batches to cache")
    else:
        print("ℹ️ No examples generated, nothing to save.")

    print("🎉 All documents processed successfully!")
    return cache

In [ ]:
cache_dir = "/content/drive/MyDrive/cache"
file_paths = ['supreme_court_1885.txt']

In [ ]:
cache = process_legal_documents_simple(file_paths,cache_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

📄 Processing supreme_court_1885.txt...


Token indices sequence length is longer than the specified maximum sequence length for this model (2913292 > 512). Running this sequence through the model will result in indexing errors


✅ Created 1459 training examples
📦 Creating batches from 1459 examples...
✅ Created 183 batches (batch_size=8)
💾 Saving all batches to cache...
💾 Saved 183 batches to batch077.pt
📊 Stats: 1459 sequences, 2988032 tokens
💾 Saved all processed batches to cache
🎉 All documents processed successfully!


## Jsonl

In [ ]:
from typing import List, Dict, Any
from transformers import RobertaTokenizer
import torch
import json

class JSONLDataProcessor:
    """
    Processes any JSONL data using RobertaTokenizer.
    Handles any JSON format by converting entire JSON to string, then chunking.
    """

    def __init__(self, tokenizer_name: str = "roberta-base", max_length: int = 512, overlap: int = 50):
        self.tokenizer = RobertaTokenizer.from_pretrained(tokenizer_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.max_length = max_length
        self.overlap = overlap

    def parse_jsonl_file(self, file_path: str) -> List[Dict]:
        """
        Parse JSONL file - reads each line as JSON object
        Returns list of raw JSON objects
        """
        json_objects = []

        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue

                try:
                    data = json.loads(line)
                    json_objects.append(data)
                except json.JSONDecodeError as e:
                    print(f"❌ Error parsing JSON at line {line_num}: {e}")
                    continue

        return json_objects

    def tokenize_json_objects(self, json_objects: List[Dict]) -> List[Dict[str, torch.Tensor]]:
        """
        Tokenize JSON objects by converting them to string and then creating
        overlapping chunks of fixed max_length.
        """
        tokenized_examples = []

        for json_obj in json_objects:
            # Convert entire JSON to string
            text = json.dumps(json_obj)

            # Tokenize the full text without padding or truncation first
            full_tokenized = self.tokenizer(
                text,
                padding=False,
                truncation=False,
                return_tensors=None # Return as list to easily slice
            )
            full_input_ids = full_tokenized['input_ids']
            # Note: Roberta tokenizer adds <s> (0) at start and </s> (2) at end automatically
            # We will handle these with slicing for chunks.

            # Implement overlapping chunking
            current_position = 0
            while current_position < len(full_input_ids):
                end_position = min(current_position + self.max_length, len(full_input_ids))

                # Extract chunk. Ensure it's exactly max_length or smaller for the last one.
                chunk_ids = full_input_ids[current_position:end_position]

                if not chunk_ids:
                    break # Should not happen if current_position is handled correctly, but safety check

                # Pad or truncate chunk_ids to max_length
                input_ids_tensor = torch.full((self.max_length,), self.tokenizer.pad_token_id, dtype=torch.long)
                attention_mask_tensor = torch.zeros((self.max_length,), dtype=torch.long)

                # Copy chunk_ids into the tensor, respecting max_length
                actual_len = min(len(chunk_ids), self.max_length)
                input_ids_tensor[:actual_len] = torch.tensor(chunk_ids[:actual_len], dtype=torch.long)
                attention_mask_tensor[:actual_len] = 1

                # Create labels by shifting input_ids
                labels = input_ids_tensor.clone()
                labels = torch.cat((labels[1:], torch.full((1,), -100, dtype=torch.long)))

                # Mask out padding tokens in labels with -100 (where attention_mask is 0)
                labels = torch.where(attention_mask_tensor == 0, torch.tensor(-100, dtype=torch.long), labels)

                example = {
                    'input_ids': input_ids_tensor,
                    'attention_mask': attention_mask_tensor,
                    'labels': labels,
                    'length': self.max_length # All examples from this method will have fixed length
                }
                tokenized_examples.append(example)

                # Move to the next chunk position
                # If the last chunk was shorter than max_length, don't move backwards with overlap
                if actual_len < self.max_length: # If it's the last, shorter chunk, stop
                    break
                current_position += (self.max_length - self.overlap)

        return tokenized_examples

    def create_smart_batches(self, examples: List[Dict[str, torch.Tensor]], tokens_per_batch: int = 3000) -> List[Dict[str, torch.Tensor]]:
        """
        Create smart batches based on token count, minimizing padding for fixed-length examples.
        """
        batches = []
        current_batch_examples = []
        current_batch_tokens = 0

        for example in examples:
            example_tokens = example['length']

            if current_batch_examples and (current_batch_tokens + example_tokens > tokens_per_batch):
                batches.append(self._stack_batch(current_batch_examples))
                current_batch_examples = []
                current_batch_tokens = 0

            current_batch_examples.append(example)
            current_batch_tokens += example_tokens

        if current_batch_examples:
            batches.append(self._stack_batch(current_batch_examples))

        return batches

    def _stack_batch(self, batch_examples: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        """
        Stacks fixed-length examples into a single batch tensor.
        Assumes all examples in `batch_examples` have the same `self.max_length`.
        """
        if not batch_examples:
            return None

        return {
            'input_ids': torch.stack([ex['input_ids'] for ex in batch_examples]),
            'attention_mask': torch.stack([ex['attention_mask'] for ex in batch_examples]),
            'labels': torch.stack([ex['labels'] for ex in batch_examples])
        }

print("JSONLDataProcessor class updated with chunking and truncation logic.")

JSONLDataProcessor class updated with chunking and truncation logic.


In [ ]:
def process_jsonl_smart(input_file: str, cache_dir: str = "./smart_cache", tokens_per_batch: int = 3000, max_length: int = 1012, overlap: int = 0):
    """
    Smart pipeline: Read JSONL -> Tokenize -> Smart Batch -> Cache
    """
    # Initialize processor and cache
    processor = JSONLDataProcessor(max_length=max_length, overlap=overlap)
    cache = SimpleBatchCache(cache_dir)

    # Read and parse JSONL file
    print(f"📖 Reading JSONL data from: {input_file}")
    json_objects = processor.parse_jsonl_file(input_file)
    print(f"✅ Found {len(json_objects)} JSON objects")

    # Tokenize examples
    print("🔤 Tokenizing examples...")
    tokenized_examples = processor.tokenize_json_objects(json_objects)
    print(f"✅ Tokenized {len(tokenized_examples)} examples")

    total_tokens = sum(ex['length'] for ex in tokenized_examples)
    print(f"📊 Total tokens: {total_tokens}")

    # Create smart batches
    print(f"📦 Creating smart batches (~{tokens_per_batch} tokens per batch)...")
    batches = processor.create_smart_batches(tokenized_examples, tokens_per_batch)

    # Calculate actual statistics
    actual_tokens_per_batch = [batch['input_ids'].numel() for batch in batches]
    avg_tokens_per_batch = sum(actual_tokens_per_batch) / len(actual_tokens_per_batch) if actual_tokens_per_batch else 0

    print(f"✅ Created {len(batches)} batches")
    print(f"📊 Average tokens per batch: {avg_tokens_per_batch:.0f}")

    # Save to cache
    print("💾 Saving batches to cache...")
    cache.save_batches(batches, {
        "source_file": input_file,
        "data_type": "jsonl_data",
        "total_batches_in_file": len(batches),
        "original_objects": len(json_objects),
        "tokenizer": "roberta-base",
        "tokens_per_batch_target": tokens_per_batch,
        "total_tokens": total_tokens,
        "max_length": max_length,
        "overlap": overlap
    })

In [ ]:
cache_dir = "/content/drive/MyDrive/cache"
input_files_to_combine = ['/content/motor_vehical_act1988_JSONL.jsonl','/content/motor_vehicle_cases (1).jsonl']
output_file = "/content/motor_vehical.jsonl"

In [ ]:
import json

def combine_jsonl_files(input_files, output_file):
    """
    Combine multiple JSONL files into a single JSONL file.

    Args:
        input_files (list): List of paths to input .jsonl files
        output_file (str): Path to the output .jsonl file
    """
    with open(output_file, 'w', encoding='utf-8') as outfile:
        for file_path in input_files:
            with open(file_path, 'r', encoding='utf-8') as infile:
                for line in infile:
                    line = line.strip()
                    if line:  # skip empty lines
                        outfile.write(line + '\n')
    print(f"✅ Successfully combined {len(input_files)} files into {output_file}")

In [ ]:
combine_jsonl_files(input_files_to_combine, output_file)

FileNotFoundError: [Errno 2] No such file or directory: '/content/motor_vehical_act1988_JSONL.jsonl'

In [ ]:
process_jsonl_smart(
        input_file=output_file,
        cache_dir=cache_dir,
        tokens_per_batch=3600
    )

## Hugging Face

In [ ]:
from datasets import load_dataset
from transformers import RobertaTokenizer
import torch

dataset = load_dataset("Alignment-Lab-AI/Lawyer-Instruct")

print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpacmygavel.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9241 [00:00<?, ? examples/s]

{'instruction': 'This problem seems like it requires a lot of dynamic reasoning and incorporating external information. I suggest using the technique of generating reasoning traces and task-specific actions in an interleaved manner. This will allow us to create and adjust plans while also interacting with external sources for additional information.', 'input': '', 'output': 'I agree that we need to be able to explore multiple reasoning paths. The technique of treating the problem as a search over a tree structure could be helpful. We can decompose the problem into intermediate steps and use a search algorithm to find the solution.'}


In [ ]:
def format_example(example):
    return f"Client: {example['instruction']}\nLawyer: {example['output']}"


In [ ]:
formatted_dataset = dataset["train"].map(lambda x: {"text": format_example(x)})


Map:   0%|          | 0/9241 [00:00<?, ? examples/s]

In [ ]:
formatted_dataset[0]

{'instruction': 'This problem seems like it requires a lot of dynamic reasoning and incorporating external information. I suggest using the technique of generating reasoning traces and task-specific actions in an interleaved manner. This will allow us to create and adjust plans while also interacting with external sources for additional information.',
 'input': '',
 'output': 'I agree that we need to be able to explore multiple reasoning paths. The technique of treating the problem as a search over a tree structure could be helpful. We can decompose the problem into intermediate steps and use a search algorithm to find the solution.',
 'text': 'Client: This problem seems like it requires a lot of dynamic reasoning and incorporating external information. I suggest using the technique of generating reasoning traces and task-specific actions in an interleaved manner. This will allow us to create and adjust plans while also interacting with external sources for additional information.\nLaw

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize_fn(example):
    # Tokenize without padding or truncation to get original lengths
    tokens = tokenizer(
        example["text"],
        truncation=False, # Do not truncate here
        padding=False     # Do not pad here
    )
    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "length": len(input_ids) # Store actual length
    }

tokenized_dataset = formatted_dataset.map(tokenize_fn)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Map:   0%|          | 0/9241 [00:00<?, ? examples/s]

In [ ]:
cache_dir = "/content/drive/MyDrive/cache0"
cache = SimpleBatchCache(cache_dir)

In [ ]:
tokenized_examples = []
for ex in tokenized_dataset:
    input_ids = torch.tensor(ex["input_ids"])
    attention_mask = torch.tensor(ex["attention_mask"])

    # Create labels by shifting input_ids to the left
    # labels[i] should be input_ids[i+1]
    labels = input_ids.clone()
    # Shift labels left by one, and set the last token to -100
    labels = torch.cat((labels[1:], torch.tensor([-100], dtype=torch.long)))

    # Ensure labels have the same length as input_ids and attention_mask
    # (This check might be redundant if HF tokenizer already padded to max_length, but safe to include)
    if len(labels) < len(input_ids):
        padding_needed = len(input_ids) - len(labels)
        labels = torch.cat((labels, torch.full((padding_needed,), -100, dtype=torch.long)))
    elif len(labels) > len(input_ids): # Should not happen
        labels = labels[:len(input_ids)]

    # Mask out padding tokens in labels with -100 (where attention_mask is 0)
    # This is the crucial step to ensure pad_token_id (1) in inputs maps to -100 in labels
    labels = torch.where(attention_mask == 0, torch.tensor(-100, dtype=torch.long), labels)

    tokenized_examples.append({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids) # Add length for smart batching
    })

In [ ]:
tokenized_examples[2]

{'input_ids': tensor([    0, 47952,    35,    20,   200,  2125,     9,   335,  2029,   201,
          5377,    15,     5,  1280,     9,   418,   145,  1240,    30,   752,
          2261,     8,   559,  1799,     6,    61,    16,    11,     5,  6685,
             4, 50118, 22532,  6426,    35,    20,   371,  2125,     9,   335,
           924,   201,    14,   194,   537, 11429,    32,    45,  8216,   203,
             7,  2261,    11,    97,   982,     6,  1135,  1408,    81,   457,
            10,   325,  1932,     4,     2]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'labels': tensor([47952,    35,    20,   200,  2125,     9,   335,  2029,   201,  5377,
            15,     5,  1280,     9,   418,   145,  1240,    30,   752,  2261,
             8,   559,  1799,     6,    61,    16,    11,

In [ ]:
from typing import List, Dict, Callable
import torch

# This defines the create_smart_batches function. It's assumed to be defined from previous steps or another cell.
# If you run into a NameError, please ensure the cell containing its definition is executed first.

def create_smart_batches(
    examples: List[Dict[str, torch.Tensor]],
    tokens_per_batch: int = 6500,
    pad_batch: Callable[[List[Dict[str, torch.Tensor]]], Dict[str, torch.Tensor]] = None
) -> List[Dict[str, torch.Tensor]]:
    """
    Create smart batches based on sequence length to minimize padding.
    Each batch will contain approximately tokens_per_batch tokens.

    Args:
        examples: List of examples, each with 'length' and tensors.
        tokens_per_batch: Target number of tokens per batch.
        pad_batch: Function to pad a batch of examples. Must accept a list of examples
                   and return a dict with 'input_ids', 'attention_mask', 'labels'.

    Returns:
        List of padded batch dictionaries.
    """
    examples.sort(key=lambda x: x['length'])

    batches = []
    current_batch = []
    current_batch_tokens = 0

    for example in examples:
        example_tokens = example['length']

        if current_batch and (current_batch_tokens + example_tokens > tokens_per_batch):
            if pad_batch:
                batches.append(pad_batch(current_batch))
            else:
                batches.append(current_batch)
            current_batch = []
            current_batch_tokens = 0

        current_batch.append(example)
        current_batch_tokens += example_tokens

    if current_batch:
        if pad_batch:
            batches.append(pad_batch(current_batch))
        else:
            batches.append(current_batch)

    return batches

In [ ]:
def _pad_batch_for_hf_dataset(batch_examples: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """
    Pad a batch to the maximum sequence length in that batch for Hugging Face examples.
    """
    if not batch_examples:
        return None

    # Find max sequence length in this batch
    max_len = max(ex['length'] for ex in batch_examples)

    batch_input_ids = []
    batch_attention_masks = []
    batch_labels = []

    for example in batch_examples:
        seq_len = example['length']
        pad_len = max_len - seq_len

        # Pad input_ids with pad_token_id
        padded_input_ids = torch.cat([
            example['input_ids'],
            torch.full((pad_len,), tokenizer.pad_token_id, dtype=torch.long)
        ])

        # Pad attention_mask with 0
        padded_attention_mask = torch.cat([
            example['attention_mask'],
            torch.zeros(pad_len, dtype=torch.long)
        ])

        # Pad labels with -100
        padded_labels = torch.cat([
            example['labels'],
            torch.full((pad_len,), -100, dtype=torch.long)
        ])

        batch_input_ids.append(padded_input_ids)
        batch_attention_masks.append(padded_attention_mask)
        batch_labels.append(padded_labels)

    return {
        'input_ids': torch.stack(batch_input_ids),
        'attention_mask': torch.stack(batch_attention_masks),
        'labels': torch.stack(batch_labels)
    }

In [ ]:
cache_dir = "/content/drive/MyDrive/cache0"
cache = SimpleBatchCache(cache_dir)

In [ ]:
# Assume _pad_batch_for_hf_dataset and tokenizer are defined in previous cells
batches = create_smart_batches(tokenized_examples, tokens_per_batch=3600, pad_batch=_pad_batch_for_hf_dataset)
total_tokens = sum(ex["length"] for ex in tokenized_examples) # Use 'length' key for total_tokens

cache.save_batches(batches, {
    "source_file": "Alignment-Lab-AI/Lawyer-Instruct",
    "data_type": "huggingface_dataset",
    "total_batches_in_file": len(batches),
    "original_objects": len(tokenized_examples),
    "tokenizer": "roberta-base",
    "tokens_per_batch_target": 3600,
    "total_tokens": total_tokens
})

💾 Saved 302 batches to batch001.pt
📊 Stats: 9241 sequences, 1072248 tokens


'/content/drive/MyDrive/cache0/batch001.pt'

In [ ]:
from typing import List, Dict, Callable
import torch

## Pdf to Text

In [ ]:
! pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00


In [ ]:


import os
from PyPDF2 import PdfReader
from typing import List

def pdf_to_text(pdf_paths: List[str], output_path: str = None, strip_newlines: bool = True):
    """
    Convert multiple PDF files into a single text file by extracting all text.

    Args:
        pdf_paths (List[str]): List of paths to the input PDF files.
        output_path (str, optional): Path to save the combined output text file.
                                     Defaults to 'combined_output.txt' if not specified.
        strip_newlines (bool, optional): If True, replaces all newline characters within
                                         extracted text from pages with a single space.
                                         Defaults to False.

    Returns:
        str: Path to the generated text file.
    """
    if not pdf_paths:
        print("No PDF files provided to convert.")
        return None

    # Default output path
    if output_path is None:
        output_path = "combined_output.txt"

    text_content = []
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    processed_pdf_count = 0
    for pdf_file_path in pdf_paths:
        if not os.path.exists(pdf_file_path):
            print(f"Warning: File not found: {pdf_file_path}. Skipping.")
            continue

        if not pdf_file_path.lower().endswith(".pdf"):
            print(f"Warning: Skipping '{pdf_file_path}' as it is not a PDF file.")
            continue

        print(f"Extracting text from: {pdf_file_path}")
        try:
            reader = PdfReader(pdf_file_path)
            for page_num, page in enumerate(reader.pages, start=1):
                text = page.extract_text()
                if text:
                    if strip_newlines:
                        text = text.replace('\n', ' ') # Replace newlines with spaces
                    text_content.append(text)
                else:
                    text_content.append(f"[Page {page_num} of {pdf_file_path} has no extractable text]")
            processed_pdf_count += 1
        except Exception as e:
            print(f"Error processing PDF '{pdf_file_path}': {e}. Skipping.")

    # Write to text file
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("".join(text_content))

    print(f"Successfully combined text from {processed_pdf_count} PDFs into {output_path}")
    return output_path

In [ ]:
import os
len(os.listdir('/content'))

288

In [ ]:
len(pdf_path)

3

In [ ]:
pdf_path = os.listdir('/content')
output_path='supreme_court_1885.txt'

In [ ]:
pdf_to_text(pdf_path,output_path)

Extracting text from: Enuga_Lakshmamma_vs_Vennapuse_Chinna_Malla_Reddy_Dead_By_on_7_February_1985_1.PDF
Extracting text from: Umesh_Chandra_Shukla_Etc_Etc_vs_Union_Of_India_Ors_on_2_August_1985_1.PDF
Extracting text from: Gram_Panchayat_Of_Village_Jamalpur_vs_Malwinder_Singh_Ors_on_9_July_1985_1.PDF
Extracting text from: Lila_Krishan_vs_Mani_Ram_Godara_Ors_on_8_May_1985_1.PDF
Extracting text from: Saboj_Kumar_Bose_vs_Kanailal_Mondal_Ors_on_6_August_1985_1.PDF
Extracting text from: Maya_Rani_Punj_vs_Commissioner_Of_Income_Tax_Delhi_on_11_December_1985_1.PDF
Extracting text from: State_Of_Orissa_Other_vs_The_Tltaghur_Paper_Mills_Company_Ltd_on_1_March_1985_1.PDF
Extracting text from: M_S_Hindustan_Gum_Ceemicals_Ltd_vs_Staie_Of_Haryana_Ors_on_19_August_1985_1.PDF
Extracting text from: The_Regional_Director_vs_Bata_Shoe_Company_P_Ltd_on_11_October_1985_1.PDF
Extracting text from: Express_Newspapers_Pvt_Ltd_Ors_vs_Union_Of_India_Ors_on_7_October_1985_1.PDF
Extracting text from: Bhagwan_Das_

'supreme_court_1885.txt'

import os


# Delete only the files listed
for file in pdf_path:
    if os.path.exists(file) and file.lower().endswith(".pdf"):
        try:
            os.remove(file)
            print(f"✅ Deleted: {file}")
        except Exception as e:
            print(f"⚠️ Could not delete {file}: {e}")
    else:
        print(f"❌ File not found or not a PDF: {file}")


In [ ]:
for file in pdf_path:
    if os.path.exists(file) and file.lower().endswith(".pdf"):
        try:
            os.remove(file)
            print(f"✅ Deleted: {file}")
        except Exception as e:
            print(f"⚠️ Could not delete {file}: {e}")
    else:
        print(f"❌ File not found or not a PDF: {file}")

❌ File not found or not a PDF: .config
✅ Deleted: N_S_K_Nayar_And_Others_vs_Union_Of_India_And_Others_on_12_December_1991_1.PDF
✅ Deleted: Jumman_Khan_vs_State_Of_U_P_on_30_November_1990_1.PDF
✅ Deleted: State_Of_Uttar_Pradesh_And_Ors_vs_Ex_Pilot_Officer_Arun_Govil_on_21_November_1989_1.PDF
✅ Deleted: Union_Of_India_And_Others_vs_Jain_Shudh_Vanaspati_on_28_November_1991_1.PDF
✅ Deleted: State_Of_Maharashtra_vs_Mahadeo_Deoman_Rai_Alias_Kalal_And_on_19_April_1990_1.PDF
✅ Deleted: State_Of_U_P_vs_Kapil_Deo_And_Another_on_21_August_1991_1.PDF
✅ Deleted: Masjid_Kacha_Tank_Nahan_vs_Tuffail_Mohammed_on_9_January_1991_1.PDF
✅ Deleted: Vishwas_Nagar_Evacuee_Plot_Purchasers_vs_Under_Secretary_Delhi_Admn_And_Others_on_27_February_1990_1.PDF
✅ Deleted: Gurbax_Singh_And_Atumal_Alias_Atma_Ram_vs_State_Of_Rajasthan_And_Others_Etc_on_29_July_1991_1.PDF
✅ Deleted: B_R_Kapoor_And_Anr_vs_Union_Of_India_Uoi_And_Ors_on_9_May_1989_1.PDF
✅ Deleted: Grih_Kalyan_Kendra_Workers__Union_vs_Union_Of_India_And_Othe

## Vlaidation

In [ ]:
cache_dir = "/content/drive/MyDrive/cache"

In [ ]:
cache = SimpleBatchCache(cache_dir)

In [ ]:
batches_from_first_file = cache.load_batches('batch069.pt')
first_batch = batches_from_first_file[0]

In [ ]:
val = first_batch

In [ ]:
print(val['input_ids'].shape)

torch.Size([2, 1012])


In [ ]:
print(val['labels'][0][40:50])

tensor([    7,     5,   461,    71, 21045,  4945,    14,     5,   744,  3645])


In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

decoded_texts = []
# Iterate through each sequence in the batch and decode it individually
tokenizer.decode(first_batch['input_ids'][0], skip_special_tokens=True))




'uly?\nAnswer: The respondent is Maj.'

In [ ]:
tokens = [    0, 47952,    35,   520,   222,    24,   283,    88,  1370,   116,1437,     2,  2589,  6426,    35,   374,   195,   212,   830,   954, 4,     2,     1,     1,     1,     1,     1,     1,     1,     1]
labels = [    47952,    35,   520,   222,    24,   283,    88,  1370,   116, 1437,     2,  2589,  6426,    35,   374,   195,   212,   830,   954, 4,     2,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100]

In [ ]:
first_batch

{'input_ids': tensor([[    5, 14097,   609,  ...,     6,  7299,   757],
         [ 1635,  2624,   270,  ...,    10,  4298,    16]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1]]),
 'labels': tensor([[14097,   609,    36,  ...,  7299,   757,  -100],
         [ 2624,   270,    18,  ...,  4298,    16,  -100]])}

## Get cache INfo

In [ ]:
cache.get_file_metadata(71)

{'filename': 'batch071.pt',
 'created_at': '2025-12-02T17:04:56.221542',
 'num_batches': 798,
 'total_sequences': 5586,
 'total_tokens': 5653032,
 'avg_sequence_length': 1012.0,
 'custom': {'source_file': '/content/wikipedia_data_Traffic_ticket.jsonl',
  'data_type': 'jsonl_data',
  'total_batches_in_file': 798,
  'original_objects': 1431,
  'tokenizer': 'roberta-base',
  'tokens_per_batch_target': 8000,
  'total_tokens': 5653032,
  'max_length': 1012,
  'overlap': 0}}

In [ ]:
cache.print_detailed_stats()

BATCH CACHE STATISTICS
Total files: 71
Total batches: 102882
Total sequences: 571,803
Total tokens: 415,526,459
Avg sequence length: 726.7
Cache directory: /content/drive/MyDrive/cache
Created: 2025-11-17T09:59:08.891893
Last updated: 2025-12-04T12:40:28.679820

File Details:
  batch001.pt: 10 batches, 149 sequences, 37250 tokens
  batch003.pt: 5 batches, 69 sequences, 35328 tokens
  batch004.pt: 21 batches, 326 sequences, 166912 tokens
  batch005.pt: 2 batches, 109 sequences, 9305 tokens
  batch006.pt: 23 batches, 358 sequences, 183296 tokens
  batch007.pt: 302 batches, 9241 sequences, 1072248 tokens
  batch008.pt: 98 batches, 1560 sequences, 798720 tokens
  batch009.pt: 118 batches, 1881 sequences, 963072 tokens
  batch010.pt: 138 batches, 2203 sequences, 1127936 tokens
  batch011.pt: 125 batches, 2000 sequences, 1024000 tokens
  batch012.pt: 168 batches, 2679 sequences, 1371648 tokens
  batch013.pt: 98 batches, 1553 sequences, 795136 tokens
  batch014.pt: 115 batches, 1827 sequences

### Inspecting Memory Usage of `batch047.pt`

This code will load `batch047.pt` and then iterate through each individual batch stored within it. For each batch, it will display the batch size, the maximum sequence length after padding, and the estimated memory usage for its tensors (`input_ids`, `attention_mask`, `labels`).

## Delete the perticular fiel

In [ ]:
cache.delete_batch_file(72)

🗑️ Successfully deleted file 'batch072.pt' and its metadata.


# Wikipedia iteratie content fetch
The user has approved the current state. I will now proceed with the plan.

The next step is to update the `WikipediaPageProcessor` class to include a method for extracting relevant internal links or categories from a fetched page to discover new topics. This is essential for the iterative topic discovery process outlined in the plan.

After defining the enhanced `WikipediaPageProcessor`, I will define a single starting Wikipedia topic and the parameters for topic discovery as per the plan.


In [ ]:
!pip install wikipedia
!pip install wikipedia-api

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=8af2199282675c014f13192ff6d69a7386731338089285e66579ae9650d0350b
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.8.1-py3-none-any.whl size=15383 sha256=dd4a6857926a279c931c925b69ec7a5a9f8f63a8e4aaec52f696aa8c924ec534
  Stored in directory: /root/.cache/pip/wheels/33/3c/79/b36253689d838af4a0539782853ac3cc38a83a6591ad570dde
Successfully built wikipedia-api


In [ ]:
import wikipediaapi
import wikipedia
import re
from typing import List, Dict, Any, Set
from datetime import datetime

class WikipediaPageProcessor:
    """
    Processes Wikipedia pages to extract high-quality, structured text content
    suitable for LLM training and to discover related topics.

    Features:
    - Extracts title, summary, and full text.
    - Structures content using explicit markers (TITLE, SUMMARY, SECTION).
    - Filters out common non-content sections like 'See also', 'References', etc.
    - Cleans whitespace and handles unicode characters.
    - Provides a minimum content length check for quality filtering.
    - Includes a method to get internal links for topic discovery.
    """

    def __init__(self,
                 user_agent: str = 'LLM-Wikipedia-Extractor/1.0 (colab@example.com)',
                 language: str = 'en',
                 min_content_length: int = 500): # Minimum length for 'content' part
        self.wiki_wiki = wikipediaapi.Wikipedia(user_agent=user_agent, language=language)
        self.min_content_length = min_content_length
        self.non_content_sections = [
            "See also", "References", "External links", "Further reading",
            "Bibliography", "Notes", "Citations", "Sources", "Footnotes",
            "Works cited", "Acknowledgments", "Appendix", "Discography",
            "Filmography", "Awards", "Publications", "Gallery", "Citations and references",
            "References and sources", "History", "Contents"
        ]

    def _clean_text(self, text: str) -> str:
        """Cleans text by normalizing whitespace and removing excessive newlines."""
        if not isinstance(text, str):
            return ""
        # Remove references like [1], [2], etc.
        text = re.sub(r'\[\d+\]', '', text)
        # Remove multiple spaces, tabs, and newlines
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _should_include_section(self, section_title: str) -> bool:
        """Checks if a section should be included based on its title."""
        return section_title not in self.non_content_sections

    def _process_section(self, section: wikipediaapi.WikipediaPageSection, level: int = 0) -> List[str]:
        """Recursively processes sections and sub-sections."""
        structured_content = []
        if self._should_include_section(section.title):
            # Add section header
            structured_content.append(f"SECTION: {section.title.upper()}\n")
            cleaned_text = self._clean_text(section.text)
            if cleaned_text:
                structured_content.append(cleaned_text + "\n\n")

            for sub_section in section.sections:
                structured_content.extend(self._process_section(sub_section, level + 1))
        return structured_content

    def extract_page_data(self, topic: str) -> Dict[str, Any]:
        """
        Extracts structured and cleaned data from a Wikipedia page.

        Args:
            topic (str): The Wikipedia page title to extract.

        Returns:
            Dict[str, Any]: A dictionary containing structured page data
                            (title, summary, content) or an error message.
                            Returns None if the page does not meet quality criteria.
        """
        page = self.wiki_wiki.page(topic)
        if not page.exists():
            print(f"❌ Page not found for topic: {topic}")
            return None

        title = self._clean_text(page.title)
        summary = self._clean_text(page.summary)

        structured_content_parts = []
        if summary:
            structured_content_parts.append(f"SUMMARY: {summary}\n\n")

        # Process main sections, skipping non-content ones
        for section in page.sections:
            structured_content_parts.extend(self._process_section(section))

        full_content = "".join(structured_content_parts).strip()

        # Quality check: ensure sufficient content
        if len(full_content) < self.min_content_length:
            print(f"⚠️ Page '{topic}' content too short ({len(full_content)} chars). Skipping.")
            return None

        # Try to get references/links using the older wikipedia library for convenience
        references_links = []
        try:
            wiki_page_old = wikipedia.page(topic, auto_suggest=False)
            if wiki_page_old.references:
                references_links.append("REFERENCES:\n")
                for ref in wiki_page_old.references:
                    references_links.append(f"- {ref}\n")
                references_links.append("\n")
            if wiki_page_old.links:
                references_links.append("RELATED_LINKS:\n")
                for link in wiki_page_old.links[:10]: # Limit links for brevity
                    references_links.append(f"- {link}\n")
                references_links.append("\n")
        except wikipedia.exceptions.PageError:
            pass # Already handled by wiki_wiki.page(topic)
        except Exception as e:
            print(f"Error fetching secondary info for '{topic}': {e}")

        # Assemble the final structured document
        final_document = {
            "TITLE": title,
            "SUMMARY": summary,
            "CONTENT": full_content,
            "METADATA": {
                "source": "Wikipedia",
                "topic": topic,
                "url": page.fullurl,
                "retrieved_at": datetime.now().isoformat()
            }
        }
        if references_links:
            final_document["REFERENCES_AND_LINKS"] = "".join(references_links).strip()

        print(f"✅ Extracted data for '{topic}' (Content length: {len(full_content)} chars).")
        return final_document

    def get_internal_links(self, topic: str, max_links: int = 5) -> Set[str]:
        """
        Extracts internal Wikipedia links from a given page, focusing on main content.

        Args:
            topic (str): The Wikipedia page title.
            max_links (int): Maximum number of unique links to return.

        Returns:
            Set[str]: A set of unique internal link titles.
        """
        page = self.wiki_wiki.page(topic)
        if not page.exists():
            return set()

        internal_links = set()
        link_count = 0

        # Iterate through links in sections, prioritizing main content
        for section in page.sections:
            if self._should_include_section(section.title):
                for link in section.links:
                    # Filter out non-article links (e.g., categories, files, disambiguation)
                    if ':' not in link.title and not link.title.startswith('List of') and not link.title.startswith('Category:'):
                        internal_links.add(link.title)
                        link_count += 1
                        if link_count >= max_links:
                            return internal_links
        return internal_links

print("WikipediaPageProcessor class defined.")

WikipediaPageProcessor class defined.


**Reasoning**:
The `WikipediaPageProcessor` class has been defined. Now, I will define the initial topic and the discovery parameters as specified in the subtask instructions.



In [ ]:
initial_topic = 'Foreign relations of India'
max_links_to_follow_per_page = 20 # Limit the number of new links from each page
target_data_size_gb = 1
max_pages_to_process = 3000 # Safet break to prevent infinite loops

print(f"Initial topic defined: '{initial_topic}'")
print(f"Discovery parameters: Max links per page = {max_links_to_follow_per_page}, Target data size = {target_data_size_gb}GB, Max pages = {max_pages_to_process}")

Initial topic defined: 'Foreign relations of India'
Discovery parameters: Max links per page = 20, Target data size = 1GB, Max pages = 5000


## Implement iterative topic discovery and size-controlled data fetching

### Subtask:
Create a new code cell to implement an iterative process that starts with the initial topic, fetches its data, extracts new related topics from its content, adds them to a queue for processing, and continues fetching until the file size approaches the target or a maximum number of pages is reached. Each successfully extracted page will be written as a JSON object to `wikipedia_data.jsonl`.


**Reasoning**:
I need to implement the iterative data fetching process as described, using the `WikipediaPageProcessor` class defined previously. This involves setting up a queue, tracking visited topics, and looping to extract content and discover new topics until specified limits are met. I will import `collections` for the `deque`.



In [ ]:
import wikipediaapi
import wikipedia
import re
from typing import List, Dict, Any, Set
from datetime import datetime

class WikipediaPageProcessor:
    """
    Processes Wikipedia pages to extract high-quality, structured text content
    suitable for LLM training and to discover related topics.

    Features:
    - Extracts title, summary, and full text.
    - Structures content using explicit markers (TITLE, SUMMARY, SECTION).
    - Filters out common non-content sections like 'See also', 'References', etc.
    - Cleans whitespace and handles unicode characters.
    - Provides a minimum content length check for quality filtering.
    - Includes a method to get internal links for topic discovery.
    """

    def __init__(self,
                 user_agent: str = 'LLM-Wikipedia-Extractor/1.0 (colab@example.com)',
                 language: str = 'en',
                 min_content_length: int = 500): # Minimum length for 'content' part
        self.wiki_wiki = wikipediaapi.Wikipedia(user_agent=user_agent, language=language)
        self.min_content_length = min_content_length
        self.non_content_sections = [
            "See also", "References", "External links", "Further reading",
            "Bibliography", "Notes", "Citations", "Sources", "Footnotes",
            "Works cited", "Acknowledgments", "Appendix", "Discography",
            "Filmography", "Awards", "Publications", "Gallery", "Citations and references",
            "References and sources", "History", "Contents"
        ]

    def _clean_text(self, text: str) -> str:
        """Cleans text by normalizing whitespace and removing excessive newlines."""
        if not isinstance(text, str):
            return ""
        # Remove references like [1], [2], etc.
        text = re.sub(r'\[\d+\]', '', text)
        # Remove pronunciation guides like 'pronunciation: [...]' or similar patterns
        text = re.sub(r'\b\w+:\s*\[[^\]]+?\]\)', '', text) # Matches patterns like 'word: [some_chars])'
        # Remove multiple spaces, tabs, and newlines
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _should_include_section(self, section_title: str) -> bool:
        """Checks if a section should be included based on its title."""
        return section_title not in self.non_content_sections

    def _process_section(self, section: wikipediaapi.WikipediaPageSection, level: int = 0) -> List[str]:
        """Recursively processes sections and sub-sections."""
        structured_content = []
        if self._should_include_section(section.title):
            # Add section header
            structured_content.append(f"SECTION: {section.title.upper()}\n")
            cleaned_text = self._clean_text(section.text)
            if cleaned_text:
                structured_content.append(cleaned_text + "\n\n")

            for sub_section in section.sections:
                structured_content.extend(self._process_section(sub_section, level + 1))
        return structured_content

    def extract_page_data(self, topic: str) -> Dict[str, Any]:
        """
        Extracts structured and cleaned data from a Wikipedia page.

        Args:
            topic (str): The Wikipedia page title to extract.

        Returns:
            Dict[str, Any]: A dictionary containing structured page data
                            (title, summary, content) or an error message.
                            Returns None if the page does not meet quality criteria.
        """
        page = self.wiki_wiki.page(topic)
        if not page.exists():
            print(f"❌ Page not found for topic: {topic}")
            return None

        title = self._clean_text(page.title)
        summary = self._clean_text(page.summary)

        structured_content_parts = []
        if summary:
            structured_content_parts.append(f"SUMMARY: {summary}\n\n")

        # Process main sections, skipping non-content ones
        for section in page.sections:
            structured_content_parts.extend(self._process_section(section))

        full_content = "".join(structured_content_parts).strip()

        # Quality check: ensure sufficient content
        if len(full_content) < self.min_content_length:
            print(f"⚠️ Page '{topic}' content too short ({len(full_content)} chars). Skipping.")
            return None

        # Assemble the final structured document (excluding METADATA as requested)
        final_document = {
            "SUMMARY": summary,
            "CONTENT": full_content
        }

        print(f"✅ Extracted data for '{topic}' (Content length: {len(full_content)} chars).")
        return final_document

    def get_internal_links(self, topic: str, max_links: int = 5) -> Set[str]:
        """
        Extracts internal Wikipedia links from a given page for topic discovery.

        Args:
            topic (str): The Wikipedia page title.
            max_links (int): Maximum number of unique links to return.

        Returns:
            Set[str]: A set of unique internal link titles.
        """
        page = self.wiki_wiki.page(topic)
        if not page.exists():
            return set()

        internal_links = set()
        link_count = 0

        # Iterate through all links available directly from the page object
        for link_title in page.links.keys(): # page.links is a dictionary where keys are link titles
            # Filter out non-article links (e.g., categories, files, disambiguation)
            if (link_title.startswith("Category:") or
                link_title.startswith("List of") or
                "(disambiguation)" in link_title or
                ":" in link_title or # Often indicates non-content pages like "Portal:..."
                link_title.startswith("File:") or # Exclude file links
                link_title.startswith("Template:") # Exclude template links
               ):
                continue
            internal_links.add(link_title)
            link_count += 1
            if link_count >= max_links:
                break
        return internal_links

print("WikipediaPageProcessor class defined.")

WikipediaPageProcessor class defined.


**Reasoning**:
The `WikipediaPageProcessor` class has been updated and the previous error should be resolved. I will now re-execute the iterative data fetching process with the corrected class.



In [ ]:
import collections
import json
from pathlib import Path

# Initialize the WikipediaPageProcessor
processor = WikipediaPageProcessor(
    user_agent='LLM-Wikipedia-Extractor/1.0 (colab@example.com)',
    min_content_length=1000
)

# Output file path
output_jsonl_file = Path("wikipedia_data_Prisons_in_India.jsonl")

# Convert target_data_size_gb to bytes
target_data_size_bytes = target_data_size_gb * 1024 * 1024 * 1024

# Initialize queue with the initial topic and tracking sets
queue = collections.deque([initial_topic])
visited_topics = {initial_topic}

current_data_size_bytes = 0
pages_processed_count = 0

print(f"Starting iterative Wikipedia data extraction for '{initial_topic}'...")
print(f"Target data size: {target_data_size_gb} GB, Max pages: {max_pages_to_process}")

with open(output_jsonl_file, "w", encoding="utf-8") as f_out:
    while queue and current_data_size_bytes < target_data_size_bytes and pages_processed_count < max_pages_to_process:
        topic = queue.popleft()
        print(f"Processing topic: {topic} (Queue size: {len(queue)}, Pages processed: {pages_processed_count}, Data size: {current_data_size_bytes / (1024*1024):.2f} MB)")

        data = processor.extract_page_data(topic)

        if data:
            json_string = json.dumps(data)
            f_out.write(json_string + '\n')
            current_data_size_bytes += len(json_string.encode('utf-8')) # Get byte size
            pages_processed_count += 1

            # Extract new links for topic discovery
            new_links = processor.get_internal_links(topic, max_links=max_links_to_follow_per_page)
            for link in new_links:
                if link not in visited_topics:
                    visited_topics.add(link)
                    queue.append(link)
        else:
            # If data extraction failed or content was too short, mark as visited anyway to avoid re-processing
            # (assuming print statements in extract_page_data handle the reason for failure)
            visited_topics.add(topic)

print(f"\n--- Extraction Summary ---")
print(f"Total pages processed: {pages_processed_count}")
print(f"Final data size: {current_data_size_bytes / (1024 * 1024 * 1024):.4f} GB")
print(f"Output file: {output_jsonl_file}")


Streaming output truncated to the last 5000 lines.
✅ Extracted data for '1991 Indian economic crisis' (Content length: 12361 chars).
Processing topic: 1973–1974 stock market crash (Queue size: 21978, Pages processed: 2873, Data size: 44.11 MB)
✅ Extracted data for '1973–1974 stock market crash' (Content length: 2119 chars).
Processing topic: 1994 Papua New Guinea financial crisis (Queue size: 21978, Pages processed: 2874, Data size: 44.11 MB)
✅ Extracted data for '1994 Papua New Guinea financial crisis' (Content length: 20731 chars).
Processing topic: 1988–1992 Norwegian banking crisis (Queue size: 21980, Pages processed: 2875, Data size: 44.13 MB)
✅ Extracted data for '1988–1992 Norwegian banking crisis' (Content length: 3088 chars).
Processing topic: 1970s energy crisis (Queue size: 21979, Pages processed: 2876, Data size: 44.14 MB)
✅ Extracted data for '1970s energy crisis' (Content length: 15928 chars).
Processing topic: 1990s Armenian energy crisis (Queue size: 21981, Pages proces

**Reasoning**:
The iterative data fetching process has completed its execution, processing 1000 pages and generating a `wikipedia_data.jsonl` file. The next step is to process and cache this generated JSONL data, as outlined in the subtask instructions. This involves calling the `process_jsonl_smart` function with the `output_jsonl_file` (wikipedia_data.jsonl), the `cache_dir`, and a `tokens_per_batch` value.



In [ ]:
process_jsonl_smart(
    input_file="/content/wikipedia_data_Prisons_in_India.jsonl",
    cache_dir="/content/drive/MyDrive/cache",
    tokens_per_batch=9000
)

📖 Reading JSONL data from: /content/wikipedia_data_Prisons_in_India.jsonl


Token indices sequence length is longer than the specified maximum sequence length for this model (4472 > 512). Running this sequence through the model will result in indexing errors


✅ Found 5000 JSON objects
🔤 Tokenizing examples...
✅ Tokenized 19479 examples
📊 Total tokens: 19712748
📦 Creating smart batches (~9000 tokens per batch)...
✅ Created 2435 batches
📊 Average tokens per batch: 8096
💾 Saving batches to cache...
💾 Saved 2435 batches to batch076.pt
📊 Stats: 19479 sequences, 19712748 tokens


# Youtube

# judgements Clean

In [ ]:
import re

def clean_constitution_text(text: str) -> str:
    """
    Cleans messy constitutional/legal text into plain English format.
    Steps:
    1. Remove amendment footnotes and citations.
    2. Strip section headers and numbering clutter.
    3. Normalize whitespace.
    4. Keep only main article text.
    """

    # Remove amendment footnotes like "Subs. by ..." or "Ins. by ..." or "Omitted..."
    text = re.sub(r'\d+\.\s*Subs.*?\)', '', text)
    text = re.sub(r'\d+\.\s*Ins.*?\)', '', text)
    text = re.sub(r'\d+\.\s*Omitted.*?\)', '', text)

    # Remove citation markers like [1], [2], [citation needed]
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\[citation needed\]', '', text, flags=re.IGNORECASE)

    # Remove excessive whitespace and line breaks
    text = re.sub(r'\s+', ' ', text).strip()



    return text


def clean_file(input_path: str, output_path: str):
    """Reads a text file, cleans it, and writes the cleaned version to another file."""
    with open(input_path, 'r', encoding='utf-8') as infile:
        raw_text = infile.read()

    cleaned_text = clean_constitution_text(raw_text)

    with open(output_path, 'w', encoding='utf-8') as outfile:
        outfile.write(cleaned_text)


In [ ]:
files=[ ]

## Refine Footnote and Citation Removal

### Subtask:
Improve the `clean_constitution_text` function to more accurately remove irrelevant procedural notes and citation markers while preserving valuable legal references.


**Reasoning**:
I need to redefine the `clean_constitution_text` and `clean_file` functions to incorporate the updated regex patterns for improved text cleaning as specified in the instructions.



In [ ]:
import re

def clean_constitution_text(text: str) -> str:
    """
    Cleans messy constitutional/legal text into plain English format by removing
    irrelevant procedural notes, problematic Unicode characters, and citation markers,
    while preserving valuable legal references.
    """
    # 1a. Remove problematic Unicode characters
    text = re.sub(r'[\uf02a\uf02d\uf0b7]', '', text)

    # 1b. Refined footnote and amendment removal
    # Removes patterns like '1. Subs. by ...', '2. Ins. by ...', '3. Omitted...', '4. w.e.f. ...', 'by the ...)'
    text = re.sub(r'\s*\d+\.\s*(?:Subs|Ins|Omitted|w\.e\.f\.|by the).*?\)', '', text, flags=re.IGNORECASE)
    # Removes patterns like '* See also arts. ...'
    text = re.sub(r'\s*\*(?:\s*See also.*?)?$', '', text, flags=re.MULTILINE)

    # 1c. Targeted removal of citation markers
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\[citation needed\]', '', text, flags=re.IGNORECASE)

    # Remove excessive whitespace and line breaks
    text = re.sub(r'\s+', ' ', text).strip()

    # Normalize article numbering (e.g., "1." → "Article 1.") - this is a general cleaning, not removal
    # Only apply if it's not already part of a sentence structure that uses numbers like '1. Case Name'
    # This part was in the original and can be kept or removed based on desired output format
    # For this subtask, focus is on removal, so I'll keep it as it was but acknowledge it's not a removal step
    text = re.sub(r'(^|\s)(\d+)\.', r' Article \2.', text)

    return text


def clean_file(input_path: str, output_path: str):
    """Reads a text file, cleans it, and writes the cleaned version to another file."""
    with open(input_path, 'r', encoding='utf-8') as infile:
        raw_text = infile.read()

    cleaned_text = clean_constitution_text(raw_text)

    with open(output_path, 'w', encoding='utf-8') as outfile:
        outfile.write(cleaned_text)


# Fine Tuning

In [ ]:
import json
import torch
from typing import List, Dict
from transformers import RobertaTokenizer


class QueryAnswerContextProcessor:
    """
    Prepares Context + Question + Answer data for causal LM fine-tuning
    with RoBERTa. Only answer tokens are used for loss; context+question masked.
    """

    def __init__(self, tokenizer_name="roberta-base", max_length=512):
        self.tokenizer = RobertaTokenizer.from_pretrained(tokenizer_name)

        # Use EOS as padding if not defined
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.max_length = max_length

    def parse_jsonl(self, file_path: str) -> List[Dict]:
        """Loads JSON or JSONL file into list of dicts."""
        data = []
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                content = f.read()
                loaded = json.loads(content)
                if isinstance(loaded, list):
                    print(f"Loaded {len(loaded)} examples from JSON array.")
                    return loaded
            except json.JSONDecodeError:
                f.seek(0)
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        data.append(json.loads(line))
                    except json.JSONDecodeError:
                        print("Skipping invalid JSON line.")
                print(f"Loaded {len(data)} examples from JSONL.")
        return data

    def build_prompt(self, item: Dict) -> (str, str):
        """Formats prompt."""
        context = ""
        if "case_name" in item:
            context += f"Case Name: {item['case_name']}\n"
        if "judgment_date" in item:
            context += f"Judgment Date: {item['judgment_date']}\n"

        input_text = f"{context}Question: {item['question']}\nAnswer:"
        output_text = f" {item['answer']}"   # answer begins after space

        return input_text, output_text

    def tokenize_examples(self, records: List[Dict]) -> List[Dict]:
        tokenized = []

        for item in records:
            input_text, output_text = self.build_prompt(item)

            input_ids = self.tokenizer.encode(input_text, add_special_tokens=False)
            output_ids = self.tokenizer.encode(output_text, add_special_tokens=False)

            combined = input_ids + output_ids
            combined = combined[:self.max_length]

            attention_mask = [1] * len(combined)

            # Mask input prompt tokens
            labels = [-100] * len(input_ids) + output_ids
            labels = labels[:self.max_length]

            padding = self.max_length - len(combined)
            if padding > 0:
                combined += [self.tokenizer.pad_token_id] * padding
                attention_mask += [0] * padding
                labels += [-100] * padding

            tokenized.append({
                "input_ids": torch.tensor(combined),
                "attention_mask": torch.tensor(attention_mask),
                "labels": torch.tensor(labels)
            })

        return tokenized

    def create_batches(self, examples, tokens_per_batch=3000):
        batches, batch, token_count = [], [], 0

        for ex in examples:
            length = len(ex["input_ids"])
            if batch and token_count + length > tokens_per_batch:
                batches.append(self.stack_batch(batch))
                batch, token_count = [], 0

            batch.append(ex)
            token_count += length

        if batch:
            batches.append(self.stack_batch(batch))

        return batches

    def stack_batch(self, batch):
        """Stacks list of dicts into tensors for training."""
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch])
        }


In [ ]:
def process_jsonl_qa(input_file, cache_dir="/content/drive/MyDrive/cache", tokens_per_batch=3000, max_length=512):
    processor = QueryAnswerContextProcessor(max_length=max_length)
    cache = SimpleBatchCache(cache_dir)

    print("📖 Reading JSONL")
    rows = processor.parse_jsonl(input_file)

    print("🔤 Tokenizing")
    tokenized = processor.tokenize_examples(rows)

    print("📦 Creating batches")
    batches = processor.create_batches(tokenized, tokens_per_batch)

    print(f"💾 Saving {len(batches)} batches to cache")
    cache.save_batches(batches, {
        "items": len(rows),
        "batches": len(batches),
        "Tokenizer": "roberta-base",
        "max_length": max_length
    })

In [ ]:
process_jsonl_qa('/content/IndicLegalQA Dataset_10K.json')

📖 Reading JSONL
Loaded 10002 examples from JSON array.
🔤 Tokenizing
📦 Creating batches
💾 Saving 2001 batches to cache
💾 Saved 2001 batches to batch068.pt
📊 Stats: 10002 sequences, 5121024 tokens
